In [ ]:
!pip install catboost
!pip install lime

In [ ]:
# =============================================================================
# HYPER PARAMETER TUNING
# =============================================================================
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint

RANDOM_STATE = 42

param_grids = {
    "Logistic Regression": {
        "clf__C"          : [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0],
        "clf__solver"     : ["lbfgs"],
        "clf__penalty"    : ["l2"],          # l1 only works with saga
        "clf__max_iter"   : [500, 1000, 2000],
    },
    "Decision Tree": {
        "clf__max_depth"        : [4, 6, 8, 10, 12, 15, None],
        "clf__min_samples_leaf" : [10, 20, 30, 50, 75, 100],
        "clf__min_samples_split": [20, 40, 60, 80, 100],
        "clf__max_features"     : ["sqrt", "log2", None],
        "clf__criterion"        : ["gini", "entropy"],
    },

    "Random Forest": {
        "clf__n_estimators"     : [100, 200, 300, 500],
        "clf__max_depth"        : [8, 10, 12, 15, None],
        "clf__min_samples_leaf" : [10, 20, 30, 50],
        "clf__max_features"     : ["sqrt", "log2", 0.3],
        "clf__min_samples_split": [20, 40, 60],
    },

    "XGBoost": {
        "clf__n_estimators"     : [200, 300, 500],
        "clf__learning_rate"    : [0.01, 0.05, 0.1, 0.2],
        "clf__max_depth"        : [4, 6, 8, 10],
        "clf__subsample"        : [0.6, 0.7, 0.8, 1.0],
        "clf__colsample_bytree" : [0.6, 0.7, 0.8, 1.0],
        "clf__reg_alpha"        : [0, 0.1, 0.5, 1.0],
        "clf__reg_lambda"       : [0.5, 1.0, 2.0, 5.0],
    },

    "LightGBM": {
        "clf__n_estimators"  : [200, 300, 500],
        "clf__learning_rate" : [0.01, 0.05, 0.1, 0.2],
        "clf__num_leaves"    : [31, 63, 127],
        "clf__max_depth"     : [6, 8, 10, -1],
        "clf__reg_alpha"     : [0, 0.1, 0.5, 1.0],
        "clf__reg_lambda"    : [0.5, 1.0, 2.0],
        "clf__min_child_samples": [20, 30, 50],
    },

    "CatBoost": {
        "clf__iterations"    : [200, 300, 500],
        "clf__learning_rate" : [0.01, 0.05, 0.1, 0.2],
        "clf__depth"         : [4, 6, 8, 10],
        "clf__l2_leaf_reg"   : [1, 3, 5, 7, 10],
        "clf__bagging_temperature": [0.0, 0.5, 1.0],
        "clf__border_count"  : [32, 64, 128],
    },

    "SVM": {
        "clf__C"        : [0.01, 0.1, 0.5, 1.0, 5.0, 10.0],
        "clf__max_iter" : [1000, 2000, 3000],
        "clf__tol"      : [1e-4, 1e-3, 1e-2],
    },
}


# ── Training loop with RandomizedSearchCV ─────────────────────────────────────

tuned_results = {}

n_iter_map = {
    "Logistic Regression" : 10,
    "Decision Tree"       : 10,
    "Random Forest"       : 10,
    "XGBoost"             : 15,
    "LightGBM"            : 15,
    "CatBoost"            : 15,
    "SVM"                 : 10,
}

for model_name, pipeline in models.items():
    print(f"\n   Tuning → {model_name} ...", end=" ", flush=True)
    t0 = time()

    rs = RandomizedSearchCV(
        estimator          = pipeline,
        param_distributions= param_grids[model_name],
        n_iter             = n_iter_map[model_name],   # ← model-specific now
        cv                 = skf,
        scoring            = "f1",
        n_jobs             = N_JOBS,
        random_state       = RANDOM_STATE,
        refit              = True,
        verbose            = 0,
        return_train_score = False,
    )

    rs.fit(X_train, y_train)
    train_time = time() - t0

    best_pipeline = rs.best_estimator_
    y_pred        = best_pipeline.predict(X_test)
    clf           = best_pipeline.named_steps["clf"]

    if hasattr(clf, "predict_proba"):
        y_prob = best_pipeline.predict_proba(X_test)[:, 1]
    elif hasattr(clf, "decision_function"):
        y_prob = best_pipeline.decision_function(X_test)

    acc     = accuracy_score(y_test, y_pred)
    prec    = precision_score(y_test, y_pred, zero_division=0)
    rec     = recall_score(y_test, y_pred, zero_division=0)
    f1      = f1_score(y_test, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_prob)

    tuned_results[model_name] = {
        "Accuracy"      : round(acc,     4),
        "Precision"     : round(prec,    4),
        "Recall"        : round(rec,     4),
        "F1-Score"      : round(f1,      4),
        "ROC-AUC"       : round(roc_auc, 4),
        "Best Params"   : rs.best_params_,
        "Train Time (s)": round(train_time, 1),
        "_pipeline"     : best_pipeline,
        "_y_pred"       : y_pred,
        "_y_prob"       : y_prob,
    }

    print(
        f"done [{train_time:.1f}s]  "
        f"F1={f1:.4f}  ROC-AUC={roc_auc:.4f}  "
        f"Best: {rs.best_params_}"
    )


# ── Post-tuning metrics — all models ─────────────────────────────────────
print("\n")
print("=" * 90)
print("  POST-TUNING MODEL COMPARISON — TASK 1: DELAY CLASSIFICATION")
print("=" * 90)

metrics_cols_tuned = [
    "Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC", "Train Time (s)"
]

tuned_compare_df = pd.DataFrame(
    {m: {k: tuned_results[m][k] for k in metrics_cols_tuned} for m in tuned_results}
).T.reset_index().rename(columns={"index": "Model"})

tuned_compare_df = tuned_compare_df.sort_values(
    "F1-Score", ascending=False
).reset_index(drop=True)
tuned_compare_df.insert(0, "Rank", tuned_compare_df.index + 1)

print(tuned_compare_df.to_string(index=False))
print("=" * 90)

best_tuned_name     = tuned_compare_df.iloc[0]["Model"]
best_tuned_pipeline = tuned_results[best_tuned_name]["_pipeline"]
best_tuned_y_pred   = tuned_results[best_tuned_name]["_y_pred"]
best_tuned_y_prob   = tuned_results[best_tuned_name]["_y_prob"]

print(f"\n   Best Tuned Model : {best_tuned_name}")
print(f"   Accuracy         : {tuned_results[best_tuned_name]['Accuracy']:.4f}")
print(f"   Precision        : {tuned_results[best_tuned_name]['Precision']:.4f}")
print(f"   Recall           : {tuned_results[best_tuned_name]['Recall']:.4f}")
print(f"   F1-Score         : {tuned_results[best_tuned_name]['F1-Score']:.4f}")
print(f"   ROC-AUC          : {tuned_results[best_tuned_name]['ROC-AUC']:.4f}")
print(f"   Best Params      : {tuned_results[best_tuned_name]['Best Params']}")
print("=" * 90)

tuned_compare_df.to_csv("outputs/task1_tuned_model_comparison.csv", index=False)
print("[\u2713] Saved → outputs/task1_tuned_model_comparison.csv")

# ── Best params — all models ─────────────────────────────────
print("\n── Best Hyperparameters — All Models ─────────────────────")
for m in tuned_results:
    print(f"\n   {m}:")
    for param, val in tuned_results[m]["Best Params"].items():
        print(f"     {param:<35} : {val}")

params_rows = [
    {"Model": m, "Best Params": str(tuned_results[m]["Best Params"])}
    for m in tuned_results
]
pd.DataFrame(params_rows).to_csv("outputs/task1_best_params.csv", index=False)
print("\n[\u2713] Saved → outputs/task1_best_params.csv")

# ── Classification report — best tuned model ───────────────────────
print(f"\n── Classification Report — {best_tuned_name} (Tuned) ─────")
print(classification_report(
    y_test, best_tuned_y_pred,
    target_names=["On-Time", "Late"],
    digits=4
))

# ── Confusion matrix — best tuned model ───────────────────────
cm = confusion_matrix(y_test, best_tuned_y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm, display_labels=["On-Time", "Late"]
)
fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title(
    f"Confusion Matrix — {best_tuned_name} (Tuned)",
    fontsize=11, fontweight="bold"
)
plt.tight_layout()
plt.savefig("outputs/task1_tuned_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.close()
print("[\u2713] Saved → outputs/task1_tuned_confusion_matrix.png")

# ── ROC curves — all tuned models ──────────────────────
PALETTE           = sns.color_palette("Set2", n_colors=len(tuned_results))
MODEL_ORDER_TUNED = tuned_compare_df["Model"].tolist()

fig, ax = plt.subplots(figsize=(8, 6))
for i, model_name in enumerate(MODEL_ORDER_TUNED):
    fpr, tpr, _ = roc_curve(y_test, tuned_results[model_name]["_y_prob"])
    auc_val     = tuned_results[model_name]["ROC-AUC"]
    ax.plot(
        fpr, tpr,
        label=f"{model_name} (AUC={auc_val:.4f})",
        color=PALETTE[i], lw=1.8
    )
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title(
    "Task 1 — ROC Curves: All Tuned Models",
    fontsize=13, fontweight="bold"
)
ax.legend(fontsize=7.5, loc="lower right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("outputs/task1_tuned_roc_curves.png", dpi=150, bbox_inches="tight")
plt.close()
print("[\u2713] Saved → outputs/task1_tuned_roc_curves.png")

# ── Heatmap — post-tuning metrics ─────────────────────────────
hm_cols = ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]
hm_df   = tuned_compare_df.set_index("Model")[hm_cols].astype(float)

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(
    hm_df,
    annot=True, fmt=".4f", cmap="YlGn",
    linewidths=0.5, ax=ax,
    cbar_kws={"shrink": 0.7},
    annot_kws={"size": 9},
)
ax.set_title(
    "Task 1 — Tuned Model Performance Heatmap",
    fontsize=13, fontweight="bold"
)
ax.set_ylabel("")
plt.tight_layout()
plt.savefig("outputs/task1_tuned_heatmap.png", dpi=150, bbox_inches="tight")
plt.close()
print("[\u2713] Saved → outputs/task1_tuned_heatmap.png")

# ── CV F1 bar chart — tuned models ───────────────────────────
# Re-run CV on the best estimator from each tuned pipeline to get
# post-tuning cross-validated F1 for reporting
print("\n── Post-Tuning Cross-Validation F1 (5-Fold) ────────────")

tuned_cv_means = []
tuned_cv_stds  = []

for model_name in MODEL_ORDER_TUNED:
    cv_s = cross_val_score(
        tuned_results[model_name]["_pipeline"],
        X_train, y_train,
        cv=skf, scoring="f1", n_jobs=N_JOBS
    )
    tuned_cv_means.append(cv_s.mean())
    tuned_cv_stds.append(cv_s.std())
    print(f"   {model_name:<22} CV F1 = {cv_s.mean():.4f} ± {cv_s.std():.4f}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(
    MODEL_ORDER_TUNED, tuned_cv_means,
    xerr=tuned_cv_stds,
    color=PALETTE[:len(MODEL_ORDER_TUNED)],
    height=0.55, capsize=5, ecolor="gray"
)
ax.set_xlabel("CV F1-Score (mean ± std)")
ax.set_title(
    "Task 1 — Cross-Validated F1 Score: Tuned Models (5-Fold)",
    fontsize=12, fontweight="bold"
)
ax.set_xlim(0, 1.05)
for i, (m, s) in enumerate(zip(tuned_cv_means, tuned_cv_stds)):
    ax.text(m + s + 0.01, i, f"{m:.4f}", va="center", fontsize=8)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("outputs/task1_tuned_cv_f1.png", dpi=150, bbox_inches="tight")
plt.close()
print("[\u2713] Saved → outputs/task1_tuned_cv_f1.png")

# ── Final summary ─────────────────────────────────────
print("\n")
print("=" * 90)
print("  FINAL SUMMARY — TASK 1: POST-TUNING RESULTS")
print("=" * 90)
print(tuned_compare_df[[
    "Rank", "Model", "Accuracy", "Precision",
    "Recall", "F1-Score", "ROC-AUC", "Train Time (s)"
]].to_string(index=False))
print("=" * 90)
print(f"\n   Best Tuned Model : {best_tuned_name}")
print(f"   Accuracy         : {tuned_results[best_tuned_name]['Accuracy']:.4f}")
print(f"   Precision        : {tuned_results[best_tuned_name]['Precision']:.4f}")
print(f"   Recall           : {tuned_results[best_tuned_name]['Recall']:.4f}")
print(f"   F1-Score         : {tuned_results[best_tuned_name]['F1-Score']:.4f}")
print(f"   ROC-AUC          : {tuned_results[best_tuned_name]['ROC-AUC']:.4f}")
print(f"\n   Saved outputs:")
print("    • outputs/task1_tuned_model_comparison.csv")
print("    • outputs/task1_best_params.csv")
print("    • outputs/task1_tuned_confusion_matrix.png")
print("    • outputs/task1_tuned_roc_curves.png")
print("    • outputs/task1_tuned_heatmap.png")
print("    • outputs/task1_tuned_cv_f1.png")
print("=" * 90)